In [1]:
import json
import torch
from transformers import pipeline

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

line_name = "BIG EDDY-OSTRAND 500KV"

base_message = [
    {
        "role": "system",
        "content": """
You extract the two endpoint bus/substation names from BPA transmission-line names.

Return ONLY valid JSON using exactly this format:

{
  "first_bus": "BUS_NAME",
  "second_bus": "BUS_NAME"
}

Rules:
- Extract the two substations or buses connected by the transmission line.
- Remove voltage levels, circuit numbers, and other equipment information.
- Preserve the actual bus/substation names.
- Do not add explanations.
- Do not use Markdown.
- Do not guess.
- If either endpoint cannot be determined, return null for that endpoint.
""",
    },
]

/home/aj/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 254/254 [00:00<00:00, 338.82it/s]


In [2]:
def get_buses_from_line_name(line_name):
    prompt = base_message + [
        {"role": "user", "content": f"Transmission line name: {line_name}"}
    ]
    output = pipe(
        prompt,
        max_new_tokens=100,
        do_sample=False,
    )
    response = output[0]["generated_text"][-1]["content"]
    buses = json.loads(response)
    return buses["first_bus"], buses["second_bus"]

In [3]:
get_buses_from_line_name("BIG EDDY-OSTRAND 500KV")

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


('BIG EDDY', 'OSTRAND')